In [0]:
from pyspark.sql.functions import current_timestamp

CATALOG = "shop_stream"
SCHEMA = "core"

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/events/orders_stream"
SCHEMA_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/events/schema/orders"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/events/checkpoints/orders"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.orders"

In [0]:
orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.rescuedDataColumn", "_rescued_data")
        .trigger(availableNow=True)
        .load(SOURCE_PATH)
)

In [0]:
orders_stream.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- status: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
query = (
    orders_stream.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .outputMode("append")
        .toTable(BRONZE_TABLE)
)